# 15.a — Ad hoc test: does the growing season bring enough heat to reach maturity? (GAEZ v4 heat units)

**Ad hoc test, outside the suitability chain**: it reads the daily ERA5 temperatures (as 1.1.d and 10.a), the calendar
and the BIO variables of 1.1.d, the delta BIO variables of the runs (1.3.a) and the final suitability of 6.b, and writes
only in `<out_root>/test_heat_sum_floor/<route>/<set>/`. Nothing of the chain is changed.

**Question.** The heat sum was tested as a predictor of the SVM in 10.a (note 8.6): redundant with the BIO means, not
retained. It was not tested as a physiological floor: a crop sown at the GGCMI date cannot reach maturity if the season
does not bring the heat its cultivars need, whatever the SVM says. Next to the ECOCROP TMIN floor (4.g), does a heat
requirement remove suitable land on ERA5, in the runs, or change the effect of the AMOC weakening? 14.a found that the
cooling of the collapse never carries today's cropland across a cold margin; the floor matters most for the land the
warming makes suitable in the north.

**Requirement (fixed before looking at the results).** GAEZ v4 model documentation, appendix table A4-6.1 (Fischer et al.
2021; the workbook `GAEZv4_Appendices.xlsx` of the project): for each crop type and growth-cycle length, a minimum
temperature (the base) and the lower boundary of the accumulated heat units, below which the crop type is not
suitable. Heat units = the sum over the cycle of the daily mean temperature on the days when it exceeds the base
(GAEZ definition). For each cell the crop type row with the cycle length nearest to the GGCMI season length is used
(clipped to the range of the table); a cell passes when **at least one** crop type of the crop passes (the most
tolerant reading, as the thresholds of 11.c):

| Crop | GAEZ crop types (base, °C) | Lower boundary of heat units by cycle length |
|---|---|---|
| Maize | temperate and subtropical (10); tropical lowland (15); tropical highland (10) | 1750 (90 d) – 2500 (180 d); 2100–2550 (90–135 d); 2000–3500 (120–300 d) |
| Soybean | temperate and subtropical (10); tropical and subtropical (12.5) | 1850–2300 (105–150 d); 2200–2700 (105–135 d) |
| Rice | japonica wetland (7.5); indica wetland (15); indica dryland (15) | 1900–2750 (105–195 d); 2500–3000 (105–150 d); 2300–2700 (105–135 d) |
| Potato | white potato (5) | 1200 (90 d) – 2100 (180 d) |
| Spring wheat | spring wheat (5); subtropical cultivars (5); tropical highland (5) | 1200–1600 (90–150 d); 1400–1700 (105–150 d); 1250–1625 (100–190 d) |
| Winter wheat | winter wheat (5); cycle = days before + after dormancy | 1300 (35+105 d) – 1600 (50+150 d) |
| Wheat | all the wheat types above | |

**Climates.** ERA5 1981–2014 (harvest years 1982–2014) daily mean temperature (Tmax + Tmin) / 2 over the GGCMI season
of each cell. The runs: the same daily series shifted by the change of the run in each phase, from the delta BIO
variables of 1.3.a (phase 1: (BIO01 + BIO02) / 2, phase 2: (BIO06 + BIO07) / 2, run − ERA5): the temperature change the
SVM and the TMIN test see. Calendar fixed (as the whole chain): a warmer season accumulates more heat over the same
dates, a colder one less.

**Outputs.** (1) Cost on ERA5: share of today's crop area (MIRCA2000) in cells suitable in 6.b that fail the floor; a
check of the definition (degree-days above the base would be far stricter). (2) Suitable land per scenario with the
floor (6.b AND heat units ≥ requirement), all land and north of 50° N, and the land the floor removes. (3) Effect of the
AMOC weakening (hosing − no hosing, cells assessable in both, as 6.b) with and without the floor. (4) Maps: margin of
heat units on ERA5, land removed in 2050.

**Reading.** A small cost on ERA5 (a few % of the area at most, as TMIN: 0–0.6 point) shows the requirement is
consistent with where the crops are grown; then the land it removes in the runs and its effect on the AMOC signal say
whether the heat requirement constrains the results (mostly the northern land gained with the warming).

Outputs in `<out_root>/test_heat_sum_floor/<route>/<set>/`: `heat_units_<crop>.nc`, `cost_era5.csv`, `areas_floor.csv`,
`amoc_effect_floor.csv`, `amoc_effect_regions_floor.csv`, the figures.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters: the final set of 1.3.a, 4.g and 6.b
CLIMATE = os.environ.get("HEAT_CLIMATE", "era5")    # daily climate of the configuration (as 1.1.d; local test: stand-in)
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")
CELL_AREA_THR = 0.0005
CROPS = os.environ.get("HEAT_CROPS", "Maize_rf,Potato_rf,Soybean_rf,Rice_rf,Wheat_rf,Wheat-winter_rf,Wheat-spring_rf").split(",")
ERA5_NAME, *ERA5_PERIOD = os.environ.get("HEAT_ERA5", "ERA5,1981,2014").split(",")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]
HU_DEF = "gaez"     # "gaez": sum of the daily mean temperature on the days above the base (GAEZ v4 heat units);
                    # "dd": degree-days above the base (computed on ERA5 too, for the check of the definition only)

# ---- GAEZ v4 model documentation, table A4-6.1 (Fischer et al. 2021): per crop type, base temperature (degC) and the
# lower boundary of the accumulated heat units (degC day) by growth-cycle length (days); below it the type is not suitable
MAIZE = {"maize temperate and subtropical": (10.0, {90: 1750, 105: 1875, 120: 2000, 135: 2125, 150: 2250, 165: 2375, 180: 2500}),
         "maize tropical lowland": (15.0, {90: 2100, 105: 2250, 120: 2400, 135: 2550}),
         "maize tropical highland": (10.0, {120: 2000, 150: 2250, 180: 2500, 210: 2750, 240: 3000, 270: 3250, 300: 3500})}
SOY = {"soybean temperate and subtropical": (10.0, {105: 1850, 120: 2000, 135: 2150, 150: 2300}),
       "soybean tropical and subtropical": (12.5, {105: 2200, 120: 2450, 135: 2700})}
RICE = {"japonica wetland rice": (7.5, {105: 1900, 120: 2100, 135: 2300, 150: 2500, 165: 2600, 180: 2700, 195: 2750}),
        "indica wetland rice": (15.0, {105: 2500, 120: 2700, 135: 2900, 150: 3000}),
        "indica dryland rice": (15.0, {105: 2300, 120: 2500, 135: 2700})}
POTATO = {"white potato": (5.0, {90: 1200, 105: 1350, 120: 1500, 135: 1650, 150: 1750, 165: 1950, 180: 2100})}
SPRING = {"spring wheat": (5.0, {90: 1200, 105: 1300, 120: 1400, 135: 1500, 150: 1600}),
          "wheat subtropical cultivars": (5.0, {105: 1400, 120: 1500, 135: 1600, 150: 1700}),
          "wheat tropical highland": (5.0, {100: 1250, 115: 1375, 130: 1500, 145: 1625, 160: 1625, 175: 1625, 190: 1625})}
WINTER = {"winter wheat": (5.0, {140: 1300, 160: 1400, 180: 1500, 200: 1600})}   # cycle 35+105 ... 50+150 days
REQ = {"Maize_rf": MAIZE, "Soybean_rf": SOY, "Rice_rf": RICE, "Potato_rf": POTATO, "Wheat-spring_rf": SPRING,
       "Wheat-winter_rf": WINTER, "Wheat_rf": {**SPRING, **WINTER}}

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
DELTA = Path(os.environ.get("HEAT_DELTA") or cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET))
B6 = Path(os.environ.get("HEAT_6B") or cu.out_dir(CFG, "6_climate_soil_amoc", ROUTE, SET))
OUT = Path(os.environ.get("HEAT_OUT") or cu.out_dir(CFG, "test_heat_sum_floor", ROUTE, SET))   # the only folder written
OUT.mkdir(parents=True, exist_ok=True)
SCEN = ["ERA5", *RUNS]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
I, J = np.nonzero(LAND)
NC = I.size
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
LAND_MHA = LAND_HA * LAND / 1e6
LAT2, LON2 = np.meshgrid(GRID["lat"], GRID["lon"], indexing="ij")
LON180 = np.mod(LON2 + 180, 360) - 180
REGIONS = {"global": LAND,                                                   # as 6.b
           "Europe": LAND & (LAT2 >= 35) & (LAT2 <= 72) & (LON180 >= -25) & (LON180 <= 45),
           "north of 50 N": LAND & (LAT2 > 50),
           "23.5 - 50 N": LAND & (LAT2 > 23.5) & (LAT2 <= 50),
           "tropics": LAND & (np.abs(LAT2) <= 23.5),
           "south of 23.5 S": LAND & (LAT2 < -23.5)}
mha = lambda a: float(np.nansum(LAND_MHA * a))
print(f"{CLIMATE} {ERA5_PERIOD[0]}-{ERA5_PERIOD[1]} | {NC} land cells | crops {CROPS} | heat units: {HU_DEF} | "
      f"written only in {OUT}")

## 1. Daily mean temperature at the land cells (as 10.a), only if heat units are to be computed

In [ ]:
def hu_file(crop):
    return OUT / f"heat_units_{crop}.nc"


def hu_attrs(crop):
    return {"crop": crop, "climate": f"{CLIMATE} {ERA5_PERIOD}", "definition": HU_DEF, "requirement": str(REQ[crop]),
            "delta": str(DELTA)}


def reusable(crop):
    if not hu_file(crop).exists():
        return False
    with xr.open_dataset(hu_file(crop)) as ds:
        return all(str(ds.attrs.get(k)) == str(v) for k, v in hu_attrs(crop).items())


TODO = [c for c in CROPS if not reusable(c)]
print(f"heat units read back: {[c for c in CROPS if c not in TODO]} | to compute: {TODO}")
if TODO:
    t0 = time.time()
    tx = cu.land_series(cu.read_climate(CFG, CLIMATE, "tasmax", GRID, ERA5_PERIOD), (I, J))
    da = cu.read_climate(CFG, CLIMATE, "tasmin", GRID, ERA5_PERIOD)
    TM = (tx + cu.land_series(da, (I, J))) / 2                     # daily mean temperature (days, land cells)
    del tx
    dates = pd.DatetimeIndex(da["time"].values).normalize()
    assert TM.shape[0] == dates.size
    print(f"{dates[0].date()} -> {dates[-1].date()}, {dates.size} days, read in {(time.time() - t0) / 60:.1f} min")

## 2. Heat units of the GGCMI season per scenario, the requirement and the floor

Harvest years and seasons as `compute_bio_variables` of 1.1.d (2-year day axis; a season crossing New Year starts in
hy − 1). For a run, the daily mean temperature of ERA5 + the change of the run in phase 1 / phase 2.

In [ ]:
def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def heat_units(sow, ph2, mat, dT, bases):
    # mean over the harvest years of the heat units of the season, per (scenario, base, definition); dT: {scenario:
    # (change of phase 1, change of phase 2)} per land cell; ERA5 also with the degree-days (check of the definition)
    yr, doy = dates.year.values, dates.dayofyear.values
    years = set(yr)
    valid = np.isfinite(sow) & np.isfinite(ph2) & np.isfinite(mat)
    wrap = valid & (mat > 365)
    s0, a0, m0 = (np.nan_to_num(x) for x in (sow, ph2, mat))
    keys = [(sc, b, HU_DEF) for sc in dT for b in bases] + [("ERA5", b, d) for b in bases for d in ("gaez", "dd") if d != HU_DEF]
    acc, cnt = {k: np.zeros(NC) for k in keys}, np.zeros(NC)
    for hy in range(int(ERA5_PERIOD[0]) + 1, int(ERA5_PERIOD[1]) + 1):
        if hy not in years:
            continue
        ok = valid & (((hy - 1) in years) | ~wrap)
        idx = np.where((yr == hy) | (yr == hy - 1))[0]
        T = (doy[idx] + np.where(yr[idx] == hy, ylen(hy - 1), 0))[:, None]
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s, a, m = s0[None] + shift, a0[None] + shift, m0[None] + shift
        p1, p2 = (T >= s) & (T < a), (T >= a) & (T <= m)
        tm = TM[idx]
        for sc, (d1, d2) in dT.items():
            ts = np.where(p1, tm + d1[None], np.where(p2, tm + d2[None], np.nan))
            for b in bases:
                for (sc_, b_, df) in keys:
                    if sc_ != sc or b_ != b:
                        continue
                    v = np.where(ts > b, ts, 0.0) if df == "gaez" else np.maximum(ts - b, 0.0)
                    acc[(sc, b, df)] += np.where(ok, np.nansum(np.where(np.isfinite(ts), v, 0.0), axis=0), 0.0)
        cnt += ok
    return {k: np.where(cnt > 0, acc[k] / np.maximum(cnt, 1), np.nan) for k in keys}


def requirement(fam, L):
    # lower boundary of the heat units of the crop-type row with the cycle length nearest to L (clipped to the table)
    cyc = np.array(sorted(fam))
    req = np.array([fam[c] for c in cyc], float)
    k = np.abs(np.nan_to_num(L)[:, None] - cyc[None]).argmin(axis=1)
    return np.where(np.isfinite(L), req[k], np.nan)


def to_grid(x, fill=np.nan):
    g = np.full(LAND.shape, fill, np.float32 if np.asarray(x).dtype.kind == "f" else np.asarray(x).dtype)
    g[I, J] = x
    return g


HU, FLOOR, MARGIN, CHECK = {}, {}, {}, []
for crop in CROPS:
    t1 = time.time()
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        sow, ph2, mat = (ds[v].values[I, J].astype(float) for v in ("sowing_doy", "phase2_start_doy", "maturity_doy"))
        e1 = ((ds["BIO01"] + ds["BIO02"]) / 2).values[I, J]
        e2 = ((ds["BIO06"] + ds["BIO07"]) / 2).values[I, J]
    L = mat - sow + 1                                              # season length (days), maturity > 365 when it wraps
    fams = REQ[crop]
    bases = sorted({b for b, _ in fams.values()})
    if crop in TODO:
        dT = {"ERA5": (np.zeros(NC), np.zeros(NC))}
        for run in RUNS:
            with xr.open_dataset(DELTA / f"BIOvariables_delta_{run}_{crop}.nc") as d:
                r1 = ((d["BIO01"] + d["BIO02"]) / 2).values[I, J]
                r2 = ((d["BIO06"] + d["BIO07"]) / 2).values[I, J]
            dT[run] = (np.nan_to_num(r1 - e1), np.nan_to_num(r2 - e2))
        H = heat_units(sow, ph2, mat, dT, bases)
        xr.Dataset({f"hu_{sc}_base{b:g}_{df}": (("lat", "lon"), to_grid(v)) for (sc, b, df), v in H.items()},
                   coords={"lat": GRID["lat"], "lon": GRID["lon"]},
                   attrs={k: str(v) for k, v in hu_attrs(crop).items()}).to_netcdf(hu_file(crop))
    with xr.open_dataset(hu_file(crop)) as ds:
        H = {}
        for v in ds.data_vars:
            sc, rest = v[3:].rsplit("_base", 1)
            b, df = rest.split("_")
            H[(sc, float(b), df)] = ds[v].values[I, J]
    # floor: at least one crop type with heat units >= its requirement; margin = best (heat units - requirement)
    req = {f: requirement(tab, L) for f, (b, tab) in fams.items()}
    for sc in SCEN:
        mg = np.nanmax(np.stack([H[(sc, b, HU_DEF)] - req[f] for f, (b, _) in fams.items()]), axis=0)
        MARGIN[(crop, sc)] = to_grid(mg)
        FLOOR[(crop, sc)] = to_grid(~(mg < 0), fill=False) & LAND        # NaN (no season) not removed by the floor
    alt = "dd" if HU_DEF == "gaez" else "gaez"
    mg_alt = np.nanmax(np.stack([H[("ERA5", b, alt)] - req[f] for f, (b, _) in fams.items()]), axis=0)
    FLOOR[(crop, "ERA5 " + alt)] = to_grid(~(mg_alt < 0), fill=False) & LAND
    HU[crop] = H
    print(f"{crop}: {(time.time() - t1) / 60:.1f} min", flush=True)

## 3. Cost on ERA5, suitable land with the floor, effect of the AMOC weakening

The floor only removes land: suitable with the floor = suitable in 6.b AND heat units ≥ requirement. Cost on ERA5 =
today's crop area (MIRCA2000) in cells suitable in 6.b that fail the floor, and all cultivated cells that fail it; the
same with the other definition of the heat units, as a check.

In [ ]:
pct = lambda a, b: 100 * (a - b) / b if b else np.nan        # as 6.b
cost, areas, effect, regions, S6 = [], [], [], [], {}
for crop in CROPS:
    with xr.open_dataset(B6 / f"climate_soil_scenarios_{crop}.nc") as d6:
        S = {lv: {s: np.nan_to_num(d6[f"suitable_{lv}"].sel(scenario=s).values) == 1 for s in SCEN} for lv in LEVELS}
        N = {s: d6["novel"].sel(scenario=s).values == 1 for s in SCEN}
    S6[crop] = (S, N)
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres = LAND & (crop_ha > CELL_AREA_THR * LAND_HA)
    share = lambda m: 100 * crop_ha[m].sum() / crop_ha.sum()
    for lv in LEVELS:
        s0 = S[lv]["ERA5"]
        row = {"crop": crop, "level": lv, "today's crop area suitable in 6.b (%)": share(s0)}
        for key, lab in (("ERA5", HU_DEF), ("ERA5 " + ("dd" if HU_DEF == "gaez" else "gaez"), "check: other definition")):
            fl = FLOOR[(crop, key)]
            row[f"{lab}: suitable crop area failing the floor (points)"] = share(s0 & ~fl)
            row[f"{lab}: cultivated area failing the floor (%)"] = share(pres & ~fl)
        cost.append(row)
        for s in SCEN:
            sf = S[lv][s] & FLOOR[(crop, s)]
            r = {"crop": crop, "scenario": s, "level": lv}
            for reg in ("global", "north of 50 N"):
                rm = REGIONS[reg]
                r[f"{reg}: suitable 6.b (Mha)"] = mha(S[lv][s] & rm)
                r[f"{reg}: with the floor (Mha)"] = mha(sf & rm)
                r[f"{reg}: removed by the floor (Mha)"] = mha(S[lv][s] & ~FLOOR[(crop, s)] & rm)
            r["today's crop area suitable, with the floor (%)"] = share(sf)
            areas.append(r)
        for hos, ref in PAIRS + [(run, "ERA5") for run in RUNS]:
            both = LAND & ~(N[hos] | N[ref])
            for floor in (False, True):
                sh = S[lv][hos] & (FLOOR[(crop, hos)] if floor else True)
                so = S[lv][ref] & (FLOOR[(crop, ref)] if floor else True)
                effect.append({"crop": crop, "pair": f"{hos} - {ref}", "level": lv, "floor": floor,
                               "suitable change, assessable in both (%)": pct(mha(sh & both), mha(so & both)),
                               "gained (Mha)": mha(sh & ~so & both), "lost (Mha)": mha(so & ~sh & both),
                               "today's crop area: change (points)": share(sh) - share(so)})
                for reg, rm in REGIONS.items():
                    ao, ah = mha(so & both & rm), mha(sh & both & rm)
                    regions.append({"crop": crop, "pair": f"{hos} - {ref}", "level": lv, "floor": floor, "region": reg,
                                    "suitable change (%)": pct(ah, ao) if ao > 0.5 else np.nan})
COST = pd.DataFrame(cost).set_index(["crop", "level"])
AREAS = pd.DataFrame(areas).set_index(["crop", "scenario", "level"])
EFF = pd.DataFrame(effect).set_index(["crop", "pair", "level", "floor"])
REG = pd.DataFrame(regions).set_index(["crop", "pair", "level", "floor", "region"])
COST.to_csv(OUT / "cost_era5.csv"); AREAS.to_csv(OUT / "areas_floor.csv")
EFF.to_csv(OUT / "amoc_effect_floor.csv"); REG.to_csv(OUT / "amoc_effect_regions_floor.csv")
with pd.option_context("display.max_rows", 300, "display.max_columns", 40, "display.width", 250):
    print("(1) cost on ERA5 (points / % of today's crop area)")
    display(COST.round(2))
    print("(2) suitable land with the floor, high input")
    display(AREAS.xs("high", level="level").round(1))
    for lv in LEVELS[::-1]:
        h, o = PAIRS[-1]
        t = EFF.xs((f"{h} - {o}", lv), level=("pair", "level"))["suitable change, assessable in both (%)"].unstack("floor")
        print(f"(3) AMOC effect 2050, {lv} input, all land: without (False) and with (True) the floor"); display(t.round(2))
        t = REG.xs((f"{h} - {o}", lv, "north of 50 N"), level=("pair", "level", "region"))["suitable change (%)"].unstack("floor")
        print(f"    north of 50 N"); display(t.round(2))

## 4. Figures

(a) Margin of the heat units on ERA5 (best crop type: heat units − requirement) over the land suitable in 6.b, per crop.
(b) Land removed by the floor in 2050 without and with the hosing. (c) Effect of the AMOC weakening in 2050 with and
without the floor.

In [ ]:
LV = "high"
fig, axes = plt.subplots(len(CROPS), 1, figsize=(11, 4.2 * len(CROPS)), squeeze=False)
for ax, crop in zip(axes[:, 0], CROPS):
    S, N = S6[crop]
    m = np.where(S[LV]["ERA5"], MARGIN[(crop, "ERA5")], np.nan)
    im = ax.pcolormesh(GRID["lon"], GRID["lat"], m, cmap="RdBu", vmin=-1000, vmax=1000, shading="nearest")
    ax.set_ylim(-60, 85); fig.colorbar(im, ax=ax, label="heat units - requirement (degC day)")
    ax.set_title(f"{crop}: margin of the heat units on ERA5, land suitable in 6.b ({LV} input); red = fails the floor", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "heat_margin_era5.png", dpi=80); plt.close(fig)

fig, axes = plt.subplots(len(CROPS), 2, figsize=(20, 4.0 * len(CROPS)), squeeze=False)
for row, crop in enumerate(CROPS):
    S, N = S6[crop]
    for ax, run in zip(axes[row], ("gcm_2050_nohosing", "gcm_2050_hosing")):
        c = np.where(~LAND, np.nan, np.where(S[LV][run] & FLOOR[(crop, run)], 1.0,
                     np.where(S[LV][run], 2.0, 0.0)))
        ax.pcolormesh(GRID["lon"], GRID["lat"], c, cmap=plt.matplotlib.colors.ListedColormap(["#eeeeee", "#1a9641", "#d7191c"]),
                      vmin=-0.5, vmax=2.5, shading="nearest")
        ax.set_ylim(-60, 85)
        ax.set_title(f"{crop}, {run}: suitable (green), removed by the heat floor (red) "
                     f"{mha(S[LV][run] & ~FLOOR[(crop, run)]):.0f} Mha", fontsize=9)
fig.tight_layout(); fig.savefig(OUT / "removed_by_floor_2050.png", dpi=70); plt.close(fig)

h, o = PAIRS[-1]
fig, axes = plt.subplots(1, 2, figsize=(14, 0.6 * len(CROPS) + 2))
for ax, (src, title) in zip(axes, [("global", "all land (assessable in both)"), ("north of 50 N", "north of 50 N")]):
    for n, floor in enumerate((False, True)):
        if src == "global":
            vals = [EFF.loc[(c, f"{h} - {o}", LV, floor), "suitable change, assessable in both (%)"] for c in CROPS]
        else:
            vals = [REG.loc[(c, f"{h} - {o}", LV, floor, src), "suitable change (%)"] for c in CROPS]
        ax.barh(np.arange(len(CROPS)) + (n - 0.5) * 0.38, vals, height=0.36, color=["#4d4d4d", "#d95f02"][n],
                label="6.b" if not floor else "with the heat floor")
    ax.axvline(0, color="grey", lw=0.8); ax.set_yticks(range(len(CROPS)))
    ax.set_yticklabels([c.replace("_rf", "") for c in CROPS]); ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
    ax.set_title(f"{title}: suitable land change (%)", fontsize=10)
axes[0].legend(fontsize=8)
fig.suptitle(f"Effect of the AMOC weakening (2050 hosing - no hosing, {LV} input) with and without the heat-unit floor", fontsize=11)
fig.tight_layout(); fig.savefig(OUT / f"amoc_effect_floor_{LV}.png", dpi=100, bbox_inches="tight"); plt.show()
print("written:", sorted(p.name for p in OUT.iterdir()))

## 5. Reading

To be written after the run on BDAP: (1) the cost on ERA5 (consistency of the GAEZ requirement with where the crops are
grown; the other definition as a check); (2) the land the floor removes per scenario, mostly north of 50° N; (3) whether
the effect of the AMOC weakening changes with the floor; (4) whether to add the floor to the climate module (as TMIN).